In [1]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

print(connection.open)

True


建立資料庫

In [2]:
database = "chapter2"
with connection.cursor() as cursor:
    sql = f"""
        CREATE DATABASE IF NOT EXISTS {database}
    """
    # 執行建立的 SQL 語句
    cursor.execute(sql)
    # 執行查看資料庫
    cursor.execute("SHOW DATABASES;")
    dbs = cursor.fetchall()
print(dbs)


[{'Database': 'chapter2'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'sys'}, {'Database': 'transaction_test'}, {'Database': 'world'}]


建立資料表

In [ ]:
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)
# 是 SQL 專使語句的老師會大寫，變數會用小寫

""" 建立 user 資料表
    id 主鍵
    name 字串 不能為空
    age 整數 
    username 字串 不能為空 必須唯一
    password 字串 不能為空
"""
with connection.cursor() as cursor:
    sql = f"""
    CREATE TABLE IF NOT EXISTS user (
        id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(200) NOT NULL,
        age INT,
        username VARCHAR(200) NOT NULL UNIQUE,
        password VARCHAR(200) NOT NULL 
    );
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    
print(tables)


[{'Tables_in_chapter2': 'user'}]


寫入資料

In [11]:
from pprint import pprint
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES ('Chrisy', 25, 'ccss004', '1234')
    """
    # valued 的字串要給單引號
    # 執行寫入的 SQL 語句
    cursor.execute(sql)
    
    # 執行查詢資料表
    cursor.execute("SELECT * FROM user;")
    # 取得查詢的所有資料
    r = cursor.fetchall()
print(r)

[{'id': 1, 'name': 'Chrisy', 'age': 25, 'username': 'ccss04', 'password': '1234'}, {'id': 6, 'name': 'Chrisy', 'age': 25, 'username': 'ccss004', 'password': '1234'}]


In [ ]:
connection.commit()
# 執行後才會真的將資料寫入
# 可以最後做，但一定要記得操作 conncection.commit()

使用另一個連線查詢資料庫

In [ ]:
connection2 = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)
with connection2.cursor() as cursor:
    cursor.execute("SELECT * FROM user;") 
    # 在執行 conncection.commit() 前， 只會存在記憶體之中
    result = cursor.fetchall()

pprint(result)

[{'age': 25,
  'id': 1,
  'name': 'Chrisy',
  'password': '1234',
  'username': 'ccss04'},
 {'age': 25,
  'id': 6,
  'name': 'Chrisy',
  'password': '1234',
  'username': 'ccss004'}]


提交資料庫變更 `conncection.commit()`
##### insert / update / delete 在執行 commit 前，資料不會真的寫入，會先暫存於記憶體。只能先確認目前的修改是不是自己需要的
##### drop 不會需要 commit 就會刪除

In [ ]:
from pprint import pprint
from pymysql.err import IntegrityError 
# 要先引入pymySQL 的模組，才可以用 IntegrityError

with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES ('Chrisy', 25, 'ccss005', '1234')
    """
    try: # 避免鍵入重複的 username 導致報錯，所以加入 try except
        cursor.execute(sql)
        
        # 提交資料庫的變更
        row = cursor.rowcount  # 異動資料的行數
        print(f"資料異動:{row}行")
        if row == 1:
            connection.commit()
    except IntegrityError: 
        print("該 user 已建立")   

該 user 已建立


更新資料

In [ ]:
with connection.cursor() as cursor:
    sql = """
        UPDATE user SET age = 20
        WHERE id = 6;
    """
    cursor.execute(sql)
    
    # 可以搭配 rowcount 屬性來檢查受影響的行數
    row = cursor.rowcount  # 異動資料的行數(可以避免忘記下 where 條件)
    print(f"資料異動:{row}行")
    if row == 1: # 若沒有改到資料 row 就不會 == 1
        # 提交資料庫的變更
        connection.commit()
    else: #確保只有該行受影響，否則 rollback(可以防止忘記下條件不小心改錯)
        connection.rollback()  # 取消上方做的任何修改(忘記 where ， rollback 可以保持資料的原樣/復原資料)
        print("未改變資料")

    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)

資料異動:1行
[{'age': 25,
  'id': 1,
  'name': 'Chrisy',
  'password': '1234',
  'username': 'ccss04'},
 {'age': 20,
  'id': 6,
  'name': 'Chrisy',
  'password': '1234',
  'username': 'ccss004'},
 {'age': 25,
  'id': 7,
  'name': 'Chrisy',
  'password': '1234',
  'username': 'ccss005'}]


刪除資料表 ( 不會需要 commit 就會刪除 )

In [2]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database='chapter2'
)

with connection.cursor() as cursor:
    sql = """
        DROP TABLE IF EXISTS user;
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES;")
    result = cursor.fetchall()

print(result)

()
